# Post Processing for MIDAA model analysis
This notebook provides the code to run post-processing analysis of the output of MIDAA as supervised or unsupervised.

The notebook has been developed with  using https://sottorivalab.github.io/midaa/scMulti_multimodal.html for instruction on MIDAA application, and LLMs to aid code writing and concept explanations where required

You should run this model on a midaa venv and with a cuda (gpu). If no gpu avaialble, just edit the loading code for the model (.pkl) to not be reliant on cuda. 

The model accepts adata and .pth and .pkl model results for analysis. The output is data analysis pipelines across:
1. Statistical evaluation metrics (ELBO, t-ratio)
2. Qualitiative evaluation pipelines (Interpreting archetypes via metadata label and gene expression patterns)


In [ ]:
#importing necessary packages
# import midaa as maa
import torch
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.sparse import issparse
import os

import pickle

In [ ]:
# Set device to ensure GPU is used if its available 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')
if device.type == 'cuda':
    torch.set_default_dtype(torch.float32)

## 1. Load data and models
Loading the adata object and chosen pretrained model.
Choose your model (i.e. number of archetypes) based on the ELBO plots generated by the MIDAA model, and load that file

In [ ]:
# Load AnnData object

#BacDrop lognormalised, balanced across treatment classes
adata_path = r"C:\Users\DG1\Desktop\DALLAB\Experimenting\Data\Antibiotic resistance\BacDrop\further processing\adata_balanced_gentamicin_distribution_min_per_bin.h5ad" #update if needed
adata = sc.read_h5ad(adata_path)
print(f'Loaded AnnData with shape: {adata.shape}') #QC, make sure it is the shape you expect 


#ALK+ tumour data object 
# adata_path = r"C:\Users\DG1\Desktop\DALLAB\Experimenting\Data\Pancreatic Data\Input_Data\Adata\TUMOUR_bivona_clean_adata.h5ad"
# adata = sc.read_h5ad(adata_path)
# print(f'Loaded AnnData with shape: {adata.shape}') #QC

This step requires CUDA - change in case gpu not avaialble to load the pkl file 

In [ ]:
# #Identifying chosne number of archetypes for reference for analysis
# # num_arch = 4 #change this

# #loading the chosen model results for analysis 

# default model paths
path = r"C:\Users\DG1\Desktop\DALLAB\Experimenting\MIDAA_model\Models\vLIVE\output\Best default models\best_midaa_result_US_5arch_1507.pkl"
name = "Unsupervised_Default"


import pickle

# Load the metadata/results dictionary
with open(path, "rb") as f: #opening in read binary mode
    result = pickle.load(f) #assiging the pretrained model to result

# Assigining the pretrained model outputs to variables
inferred_quantities = result["inferred_quantities"]  # includes A, B, Z matrices
ELBO = result["ELBO"]
hyperparameters = result["hyperparameters"]

# Accessing the output matrices
A = inferred_quantities["A"]
B = inferred_quantities["B"]
Z = inferred_quantities["Z"]
archetypes_inferred = inferred_quantities["archetypes_inferred"]

## 2. Statistical Evaluation Metrics

2.a ELBO value

Provides an indication if the model would benefit from training over longer epochs / change to learning rate to improve perofrmance 

In [ ]:
#plot ELBO loss over training for the model
final_elbos = {i+1: elbo for i, elbo in enumerate(ELBO)}
plt.figure(figsize=(8, 5))
plt.plot(list(final_elbos.keys()), list(final_elbos.values()), marker='o')
plt.xlabel('Number of epochs')
plt.ylabel('ELBO for {num_arch} archetypes')
plt.title('ELBO loss over training')
plt.show()

In [ ]:
#printing final value for reference 
final_elbo_value = ELBO[-1] if isinstance(ELBO, (list, np.ndarray)) else ELBO
print("Final ELBO value:", final_elbo_value)


### 2.b t-ratio
Assessing ratio of poltyope volume vs convex hull of data in latent space


Requires: 


    1. Calulating archetype coordinates in latent space
    2. Calculating simplex volume in latent space 
    3. calculating original data volume in latent space
    4. calculate real t-ratio

In [ ]:
import numpy as np


#1. calculating archetype coordinates in latent space

archetypes_latent = archetypes_inferred
print("Archetype coordinates in latent space (shape: n_archetypes x latent_dim):")
print(archetypes_latent)
print(archetypes_latent.shape) #should be num of archetypes x latent dim, where latent dim is num of arhcetypes - 1



In [ ]:
#2. compute volume of simplex in latent space using the coordinates

from math import factorial

#structure matches ParTI method of calculating the volume in latent space

ref = archetypes_latent[-1] #using last archetype coordinates as reference
edge_vectors = (archetypes_latent[:-1] - ref).T #calculates the edge vectors of the simplex

simplex_volume = np.abs(np.linalg.det(edge_vectors)) / factorial(archetypes_latent.shape[0] - 1) #same method as ParTI for calculating the volume of the simplexin n-dimensional space



In [ ]:
#3. calculate volume of original data in latent space
from scipy.spatial import ConvexHull

hull = ConvexHull(Z)  #leveraging scipy's convex hull function for Z 
convex_hull_volume = hull.volume

In [ ]:
#4. calculate real t-ratio
t_ratio = simplex_volume / convex_hull_volume
print ("t_ratio: ", t_ratio)

## 2. Biological Interpretation of archetypes 
Characterising the archetypes by their gene expression mapped to function.

The non-linear nature of the MIDAA representation needs to be taken into account - future work should prioritise de

#### Visualizing Archetypes by Treatment and Replicate

Using built in midaa function plot_archetypes_simplex()

In [ ]:
import pandas as pd
from midaa import plot_archetypes_simplex

# Get archetype assignments (A matrix) and metadata
obs = adata.obs

# # Visualize by treatment
# fig, ax = maa.plot_archetypes_simplex(
#     result,
#     color_by=adata.obs["cell_type_final"],
#     l_title="Treatment",
#     cmap = "Set1"
# )
# fig.suptitle("Archetype Simplex by Treatment")
# fig.show()

# Visualize by replicate
fig, ax = maa.plot_archetypes_simplex(
    result,
    color_by=adata.obs["Treatement.Timepoint"],
    l_title="Timepoint",
    cmap = "Set1"
)

fig.suptitle("Archetype Simplex by Treatment Timepoint")
fig.show()

#### Plotting archetype contribution by treatment 
Developed own pipeline for quantitative evaluation

In [ ]:
#definitng a functino to understand how much each archetype contributes to each treatment label 

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

#getting dfs for treatment labels per cell ID in original space   
treatment_df= adata.obs[["treatment"]].copy() # creating a dataframe with the treatment labels and cell ids

print(treatment_df.head()) #QC

    

In [ ]:
n_archetypes = A.shape[1] 
df = pd.DataFrame(A, columns=[f"Archetype {i+1}" for i in range(n_archetypes)]) 
print(df.head())
    

Given that A is the cell x archetype, then it should align with the order that the treatment labels are in. Therfoe, the treatmetn columns can be added as an additional column in the df

In [ ]:
df["treatment"] = treatment_df["treatment"].values
print(df.head())

In [ ]:
  # Calculate mean contribution per archetype per treatment
mean_contrib = df.groupby(["treatment"]).mean() #calculates the mean weights of cells wiht a treatment to each archetype
print(mean_contrib)

In [ ]:
# # Filter for PD rows
# pd_df = df[df["treatment"] == "PD"]

# # Plot the distribution of archetype weights for PD cells
# plt.figure(figsize=(8, 5))
# for i in range(n_archetypes):
#     plt.hist(
#         pd_df[f"Archetype {i+1}"], 
#         bins=20, 
#         alpha=0.6, 
#         label=f"Archetype {i+1}"
#     )

# plt.xlabel("Archetype Weight")
# plt.ylabel("Number of PD Cells")
# plt.title("Distribution of Archetype Weights for PD Cells")
# plt.legend()
# plt.tight_layout()
# plt.show()


In [ ]:
#plot mean contribution per archetype per treatment

ax = mean_contrib.plot(kind="bar", figsize=(8, 5))
ax.set_ylabel("Mean Archetype Contribution")
ax.set_xlabel("Treatment")
ax.set_title("Archetype Contribution by Treatment")
plt.legend(title="Archetype")
plt.tight_layout()
plt.show()

#### Plotting treatment contirbution to each archetype 
Understand how much each treatment contirbutes to each archetype

In [ ]:
ax = mean_contrib.T.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Mean Treatment Contribution")
ax.set_xlabel("Archetype")
ax.set_title("Treatment Contribution by Archetype")
plt.legend(title="Treatment")
plt.tight_layout()
plt.show()

### Visualising plots of polytope in latent space

Visualising the generated polytope and input data in latent space facilitates qualitative comparison between models for biolgoical interpretation of:

1) the latent representations of the data 
2) the position of polytope within the convex hull of the data
3) the position of cells wrt to archetypes

This code leverages the Z matrix for cell coordinates and archetypes_inferred for archetype coordinates, both in the latent space

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

#accessing the coordinates of the archetypes for 2D and 3D plots in latent space 
archetypes_latent = archetypes_inferred
archetypes_latent_2d = archetypes_latent[:, :2] #accessing the first 2 latent dimensions of the archetypes
archetypes_latent_3d = archetypes_latent[:, :3] #accessing the first 2 latent dimensions of the archetypes

#accessing the coordinates of the archetypes for 2D and 3D plots in latent space 
data_latent = Z
data_latent_2d = data_latent[:, :2]
data_latent_3d = data_latent[:, :3]

#function to plot the convex hull of the data in latent space


#function to plot the polytope in latent space based on the archetype coordinates ina  similar manner to the output of ParTI 
def simplex_plot(ax, archetype_coordinates, edgecolor='blue', label='Simplex', linewidth=2):
    # ax: matplotlib axis (2D or 3D)
    # archetype_coordinates: np.ndarray, shape (n_archetypes, 2 or 3)
  
    n = archetype_coordinates.shape[0]
    dim = archetype_coordinates.shape[1]
    for i in range(n):
        for j in range(i+1, n):
            if dim == 2:
                ax.plot(
                    [archetype_coordinates[i, 0], archetype_coordinates[j, 0]],
                    [archetype_coordinates[i, 1], archetype_coordinates[j, 1]],
                    color=edgecolor, lw=linewidth, label=label if i == 0 and j == 1 else None
                )
            elif dim == 3:
                ax.plot(
                    [archetype_coordinates[i, 0], archetype_coordinates[j, 0]],
                    [archetype_coordinates[i, 1], archetype_coordinates[j, 1]],
                    [archetype_coordinates[i, 2], archetype_coordinates[j, 2]],
                    color=edgecolor, lw=linewidth, label=label if i == 0 and j == 1 else None
                )
            else:
                raise ValueError("archetype_coordinates must have 2 or 3 columns for 2D or 3D plotting.")

#plotting the data in latent space colored by treatment
def plot_data_by_treatment(ax, data_latent, treatments, treatment_to_color=None, s=5, alpha=0.5):
   
    #     ax: matplotlib axis (2D or 3D)
    #     data_latent: np.ndarray, shape (n_samples, 2 or 3)
    #     treatments: array-like, treatment labels for each sample
    #     treatment_to_color: dict, optional, mapping treatment name to color
    #     s: marker size
    #     alpha: marker transparency

    #creating a dictionary to map treatment names to colors
    unique_treatments = pd.unique(treatments)
    if treatment_to_color is None:
        treatment_to_color = {
            # "PD": "#2ECC71",  # softer green (emerald)
            # "RD": "#9B59B6",     # amethyst purple
            # "TN": "#E74C3C"       # soft red (alizarin)

            "ciprofloxacin": "#2ECC71",  # softer green (emerald)
            "gentamicin": "#9B59B6",     # amethyst purple
            "meropenem": "#E74C3C"       # soft red (alizarin)
        }
    for t in unique_treatments:
        if t not in treatment_to_color:
            treatment_to_color[t] = "#A9A9A9"  # dark gray

    #plotting the data in latent space colored by treatment
    for t in unique_treatments:
        idx = (treatments == t)
        if data_latent.shape[1] == 2:
            ax.scatter(
                data_latent[idx, 0], data_latent[idx, 1],
                s=s, color=treatment_to_color[t], alpha=alpha, label=str(t)
            )
        elif data_latent.shape[1] == 3:
            ax.scatter(
                data_latent[idx, 0], data_latent[idx, 1], data_latent[idx, 2],
                s=s, color=treatment_to_color[t], alpha=alpha, label=str(t)
            )
        else:
            raise ValueError("data_latent must have 2 or 3 columns for 2D or 3D plotting.")



Plotting the 2D plot of data and polytope in latent space

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

plot_data_by_treatment(ax, data_latent_2d, adata.obs["treatment"].values, s=5)

# Plot the simplex (archetype connections)
simplex_plot(ax, archetypes_latent_2d, edgecolor='blue', label='Simplex', linewidth=2)
# Plot the archetype vertices as white circles with black border and number them inside
ax.scatter(
    archetypes_latent_2d[:, 0], archetypes_latent_2d[:, 1],
    color='white', edgecolor='black', s=120, marker='o', label='Archetypes', zorder=5
)
for i, (x, y) in enumerate(archetypes_latent_2d):
    ax.text(
        x, y, f'{i+1}', fontsize=10, color='black', ha='center', va='center', fontweight='bold', zorder=6
    )

# Make a legend for treatments and archetypes (avoid duplicate labels)
handles, labels = ax.get_legend_handles_labels()
from collections import OrderedDict
by_label = OrderedDict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title="Treatment / Archetypes")

ax.set_title(f'{name}: Latent Space, Data, and Simplex (colored by treatment)')
plt.xlabel('Latent dim 1')
plt.ylabel('Latent dim 2')
plt.tight_layout()
plt.show()

Plotting in 3D

In [ ]:
from matplotlib.lines import Line2D
from itertools import combinations

# Create 3D figure and an axes object
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection='3d')

plot_data_by_treatment(ax, data_latent_3d, adata.obs["treatment"].values, s=5)

# Plot polytope
simplex_plot(ax, archetypes_latent_3d, edgecolor='blue', label='Simplex', linewidth=2)
# Plot the archetype vertices as stars
# Plot the archetype vertices as red stars and number them inside (3D)
ax.scatter(
    archetypes_latent_3d[:, 0], archetypes_latent_3d[:, 1], archetypes_latent_3d[:, 2],
    color='white', edgecolor='black', s=120, marker='o', label='Archetypes', zorder=5
)
for i, (x, y, z) in enumerate(archetypes_latent_3d):
    ax.text(
        x, y, z, f'{i+1}', fontsize=10, color='black', ha='center', va='center', fontweight='bold', zorder=6
    )

# Make a legend for treatments and archetypes (avoid duplicate labels)
handles, labels = ax.get_legend_handles_labels()
from collections import OrderedDict
by_label = OrderedDict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title="Treatment / Archetypes")

ax.set_title(f'{name}: Latent Space, Data, and Simplex (colored by treatment)')
plt.xlabel('Latent dim 1')
plt.ylabel('Latent dim 2')
plt.tight_layout()
plt.show()

# Distanced based analysis
aligns with ParTI pipeline of analysis

calculates the distance between each cell and a given arhcetype in latent space. Relies on arhcetype coordinates and cell coordinates in latent space
leverage scipy package for gene analysis

leverage Aria's code for distance analysis plotting functions


In [ ]:
archetypes_latent = archetypes_inferred  # accessing the coordinates of the archetypes in latent space
data_latent = Z  # accessing the coordinates of the cells in latent space

import pandas as pd
import numpy as np
from scipy.spatial import distance  # using scipy package to calculate distances between cells and archetypes in latent space

# Result: distances (n_cells, n_archetypes)
distances = distance.cdist(data_latent, archetypes_latent, metric='euclidean')

# Use adata.obs_names as the index for the distances DataFrame
distances_df = pd.DataFrame(
    distances,
    index=adata.obs_names,  # This will give row names like in your screenshot
    columns=[f"Archetype {i+1}" for i in range(archetypes_latent.shape[0])]
)

distances_df.head()



In [ ]:
# Join metadata (adata.obs) to distances_df
distances_with_meta = distances_df.join(adata.obs)

# Optionally, display the head of the new dataframe
distances_with_meta.head()


In [ ]:
def density_from_archtype(arc, treatments, c=None):
    '''
    Makes a density plot for all conditions based on distance to one archetype.
    '''
    if isinstance(treatments, str):
        treatments = [treatments]
    num_treatments = len(treatments)
    if c is None:
        # Use matplotlib default color cycle if not provided
        import itertools
        color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
        c = [color_cycle[i % len(color_cycle)] for i in range(num_treatments)]
    plt.figure(figsize=(8, 5))
    for i, treatment in enumerate(treatments):
        mask = combined_df["treatment"] == treatment
        # Use the correct column for the given archetype
        col = str(arc)
        if col in combined_df.columns:
            data = combined_df.loc[mask, col]
        else:
            # Fallback to the second column if not found (legacy behavior)
            data = combined_df.loc[mask].iloc[:, 1]
        plt.hist(data, bins=10, color=c[i], alpha=0.5, label=str(treatment))
    plt.title(f'Density from Archetype {arc} for all treatments')
    plt.xlabel('Distance from Archetype')
    plt.ylabel('Frequency')
    plt.legend()
    plt.show()

In [ ]:
def density_from_archtype_multiple_conditions(arc, treatment, c=None):
    '''
    Makes a density plot for multiple condition based on distance to one archetype
    '''
    num_treatments = len(treatment)
    if not c:
        c = ['#1f77b4'] * num_treatments
            
    fig, axes = plt.subplots(num_treatments, 1, sharey=True, figsize=(8, num_treatments * 5))

    #handle case where there is only one condition
    if num_treatments == 1:
        axes = [axes]
    
    for i, cond in enumerate(treatments):
        ax = axes[i]
        ax.hist(combined_df[combined_df["treatment"] == cond].iloc[:, 1], bins=10, color=c[i])
        ax.set_title(f'Distance from Archetype {arc} for {cond}')
        ax.set_xlabel('Distance from Archetype')
        ax.set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_density_by_arch_distance_multiple_conditions(combined_df, num_arch, treatments, c=None):
    # Makes a density plots (one plot per archetype). All treatments are plotted
    num_treatments = len(treatments)
    if not c:
        c = ['#1f77b4'] * num_treatments
    
    # two columns plot
    ncols = 2
    nrows = (num_arch) // ncols if num_arch % 2 == 0 else (num_arch) // ncols + 1
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 5), sharey=True)
    
    # Flatten axes to make indexing easier
    axes = axes.flatten()
    
    for i in range(1, num_arch + 1): #loop through each archetype
        ax = axes[i - 1]  # axes indicies are one off for archetypes 
        
        # Plot for each condition on the same subplot
        for j, treatment in enumerate(treatments): #looping through each treatment
            col = f"Archetype {i}"
            #filters for the treatment, selects distance byn archetype column, computes a 10 bin histogram of distances, n is density per bin, x is bin edges 
            n, x = np.histogram(
                combined_df.loc[combined_df["Treatement.Timepoint"] == treatment, col],
                bins=10, density=True)
            bin_centers = 0.5 * (x[1:] + x[:-1]) #convert bin edges to bin centers
            ax.plot(bin_centers, n, linestyle='-', label=treatment, color=c[j]) #plot density curve on the subplot using the associated colour with the treatment
        
        # Set titles and labels for each subplot
        ax.set_title(f"Distance from Archetype {i}")
        ax.set_xlabel("Distance from Archetype")
        ax.set_ylabel("Density")
        ax.legend(title="treatment")

    # Hide any empty subplots, this occurs if the num_arch is odd
    for i in range(num_arch, len(axes)):
        axes[i].axis('off')
    
    # Adjust layout to avoid overlapping
    plt.tight_layout()
    plt.show()

In [ ]:
def get_arch_col(df, k):
                for c in df.columns:
                    if str(c).strip().lower() == f"archetype {k}".lower():
                        return c
                raise KeyError(f"No column for Archetype {k}. Got: {list(df.columns)}")

In [ ]:
def plot_density_transition_multiple_conditions(combined_df, arch, num_arch, treatments, c=None):
    num_treatments = len(treatments)
    if not c:
        c = ['#1f77b4'] * num_treatments
    
    # two columns plot
    ncols = 2
    nrows = (num_arch-1) // ncols if (num_arch-1) % 2 == 0 else (num_arch-1) // ncols + 1
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 5), sharey=True)
    
    # Flatten axes to make indexing easier
    axes = axes.flatten()

    arch_to_plot = list(range(1, num_arch+1))
    arch_to_plot.remove(arch)
    
    for i, a in enumerate(arch_to_plot):
        ax = axes[i]  # axes indicies are one off
        for j, treatment in enumerate(treatments): #looping through each treatment
           subset = combined_df[combined_df["Treatement.Timepoint"] == treatment]
           c_arch = get_arch_col(subset, arch)
           c_a    = get_arch_col(subset, a)
           vals = np.log(np.log(subset[c_arch].astype(float).clip(lower=1e-12))) \
                - np.log(subset[c_a].astype(float).clip(lower=1e-12))
           n, x = np.histogram(vals.dropna(), bins=10, density=True)
            # Plot for each condition on the same subplot
        
           bin_centers = 0.5 * (x[1:] + x[:-1])
           ax.plot(bin_centers, n, linestyle='-',label=treatment, color=c[j])
        
        # Set titles and labels for each subplot   
        ax.set_title(f"Ratio of Distance from Archetype {arch} to {a}")
        ax.set_xticks([bin_centers[0], bin_centers[-1]], [f'Arch {arch}', f'Arch {a}'])
        ax.set_ylabel("Density")
        ax.legend(title="Treatment")

    # Hide any empty subplots, this occurs if the num_arch is odd
    for i in range(num_arch-1, len(axes)):
        axes[i].axis('off')
    
    # Adjust layout to avoid overlapping
    plt.tight_layout()
    plt.show()

In [ ]:
## colors for each condition
color_map = {
    "PD": "#2ECC71",  # softer green (emerald)
    "RD": "#9B59B6",     # amethyst purple
    "TN": "#E74C3C"       # soft red (alizarin)
}

In [ ]:
plot_density_by_arch_distance_multiple_conditions(distances_with_meta, num_arch=4, 
                                                  treatments=sorted(color_map.keys()), 
                                                  c=[color_map[key] for key in sorted(color_map.keys())])

In [ ]:
plot_density_transition_multiple_conditions(combined_df = distances_with_meta, arch=1, num_arch=5, treatments=sorted(color_map.keys()))

## Further Analysis: Genetic/Marker Analysis

Built a new function for genetic analysis. Identifies the top genes per archetype

#### Ranking gene expression contribution to archetype

In [ ]:
#loading pretrained models 
#finetuned model paths
# path = r"C:\Users\DG1\Desktop\DALLAB\Experimenting\MIDAA_model\Models\vLIVE\output\best optimised models\best_midaa_result_5arch_S_MULT_8_0108.pkl"
# name = "Supervised_FT"

# path = r"C:\Users\DG1\Desktop\DALLAB\Experimenting\MIDAA_model\Models\vLIVE\output\best optimised models\best_midaa_result_5arch_US_MULT_8_0108.pkl"
# name = "Unsupervised_FT"


#default model paths
# path = r"C:\Users\DG1\Desktop\DALLAB\Experimenting\MIDAA_model\Models\vLIVE\output\Best default models\best_midaa_result_US_5arch_1507.pkl"
# name = "Unsupervised_Default"

path = r"C:\Users\DG1\Desktop\DALLAB\Experimenting\MIDAA_model\Models\vLIVE\output\Best default models\supervised_best_midaa_result_5arch_1607.pkl"
name = "Supervised_Default"


import pickle

# Load the metadata/results dictionary
with open(path, "rb") as f: #opening in read binary mode
    result = pickle.load(f) #assiging the pretrained model to result

# Assigining the pretrained model outputs to variables
inferred_quantities = result["inferred_quantities"]  # includes A, B, Z matrices
ELBO = result["ELBO"]
hyperparameters = result["hyperparameters"]

# Accessing the output matrices
A = inferred_quantities["A"]
B = inferred_quantities["B"]
Z = inferred_quantities["Z"]
archetypes_inferred = inferred_quantities["archetypes_inferred"]

## Visualing relative gene contribution to arhcetypes based on MIDAA analysis

We need the archetype weight by genes. This is not directly availabel as an output from MIDAA model, so is created through B @ X, where B is archetype x cell matrix (input is weight in latent space) and X is cell x gene matrix (input is expression). Despite the fact B was created by the encoder i.e. latent space, it is still appropriate to multiply with X in gene space to infer the gene x archetype weights.
 

In [ ]:
#X is the cell x gene matrix - expression data in gene space per sample
X = adata.X 
print("X shape:", X.shape)

#b is the archetype X CELL matrix - description of arhcetypes by samples
B = result['inferred_quantities']['B']
print("B shape:", B.shape)


#this gives the RefSeq IDs of the genes in th eorder they appear in the X matrix 
var_names = adata.var_names
print("var_names shape:", var_names.shape if hasattr(var_names, "shape") else len(var_names))
print("var_names head:", var_names[:5] if hasattr(var_names, "__getitem__") else var_names)


In [ ]:
#in case you want to identify differntiall expreessed genes, z-score the data

# X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8) #Z score 
# archetype_refseqID = B @ X
# print(archetype_refseqID.shape) #QC to check dimensinos, should be archetypes x genes 

In [ ]:
# Plot all genes (x: gene index, y: weight, lines: archetypes)
plt.figure(figsize=(18, 6))

# Now archetype_refseqID is shape (n_archetypes, n_genes)
for i in range(archetype_refseqID.shape[0]):  # loop through each archetype (row)
    plt.plot(
        range(archetype_refseqID.shape[1]),   # gene indices
        archetype_refseqID[i, :],             # weights for all genes for this archetype
        label=f"Archetype {i+1}",
        linewidth=2,
        alpha=0.5,
    )

plt.xlabel("Gene Index")
plt.ylabel("Archetype Weight")
plt.title(f"{name} Gene Weights Across Archetypes")
plt.legend()
plt.tight_layout()
plt.show()

### Plotting all the gene expression changes per archetype

In [ ]:
# For each archetype, plot the gene weights from highest to lowest (sorted), on separate subplots but in one image.
# Keep the y-axis the same across all subplots.

# archetype_refseqID is now (n_archetypes, n_genes)
n_archetypes = archetype_refseqID.shape[0]
n_cols = min(3, n_archetypes)
n_rows = (n_archetypes + n_cols - 1) // n_cols

# Find global min and max for y-axis to keep it constant across all subplots
global_min = np.min(archetype_refseqID)
global_max = np.max(archetype_refseqID)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows), sharey=True)
axes = np.array(axes).reshape(-1)  # flatten in case of 1 row

for i in range(n_archetypes):
    ax = axes[i]
    # For each archetype, get the gene weights and sort them descending
    weights = archetype_refseqID[i, :]
    sorted_weights = np.sort(weights)[::-1]
    ax.plot(range(len(sorted_weights)), sorted_weights, linewidth=2)
    ax.set_title(f"Archetype {i+1}")
    ax.set_xlabel("Number of genes with expression weight")
    if i % n_cols == 0:
        ax.set_ylabel("Archetype Weight")
    ax.set_ylim(global_min, global_max)

# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

fig.suptitle(f"{name} Frequency of expression weight of genes per archetype", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

PLOT TOP GENES FOR EACH ARCHETYPE

In [ ]:
# Updated for archetype_refseqID shape: (n_archetypes, n_genes)
def plot_neg_pos_genes_per_archetype(archetype_refseqID, var_names, n_top):
    import math

    # archetype_refseqID is now (n_archetypes, n_genes)
    n_archetypes = archetype_refseqID.shape[0]
    n_cols = 2
    n_rows = math.ceil(n_archetypes / n_cols)

    # First, collect all values to determine global x-axis limits
    all_values_list = []
    for i in range(n_archetypes):
        weights = archetype_refseqID[i, :] if not hasattr(archetype_refseqID, "iloc") else archetype_refseqID.iloc[i, :].values
        idx_pos = np.argsort(-weights)[:n_top]
        idx_neg = np.argsort(weights)[:n_top]
        pos_values = weights[idx_pos]
        neg_values = weights[idx_neg]
        all_values = np.concatenate([pos_values, neg_values])
        all_values_list.append(all_values)
    all_values_concat = np.concatenate(all_values_list)
    x_min = np.min(all_values_concat)
    x_max = np.max(all_values_concat)
    # Add a small margin
    x_margin = 0.05 * (x_max - x_min) if x_max > x_min else 1
    x_lim = (x_min - x_margin, x_max + x_margin)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10 * n_cols, 5 * n_rows), sharex=True)
    axes = np.array(axes).reshape(-1)  # flatten in case axes is 2D

    for i in range(n_archetypes):
        ax = axes[i]
        # Get the weights for this archetype
        weights = archetype_refseqID[i, :] if not hasattr(archetype_refseqID, "iloc") else archetype_refseqID.iloc[i, :].values
        # Get indices of top n_top positive and top n_top negative genes
        idx_pos = np.argsort(-weights)[:n_top]
        idx_neg = np.argsort(weights)[:n_top]
        # Get gene names
        pos_gene_names = [var_names[j] for j in idx_pos]
        neg_gene_names = [var_names[j] for j in idx_neg]
        # Get association values
        pos_values = weights[idx_pos]
        neg_values = weights[idx_neg]
        # Combine for plotting (positive first, then negative)
        all_gene_names = pos_gene_names + neg_gene_names
        all_values = np.concatenate([pos_values, neg_values])
        # Colors: blue for positive, pink for negative
        colors = ['#4C9ED9'] * n_top + ['#C97B9B'] * n_top

        bars = ax.barh(range(2 * n_top), all_values, color=colors, alpha=0.85)
        # Add value labels to bars
        for j, bar in enumerate(bars):
            ax.text(bar.get_width() + 0.02 * np.sign(bar.get_width()), bar.get_y() + bar.get_height()/2,
                    f"{all_values[j]:.3f}", va='center', ha='left' if all_values[j] >= 0 else 'right', fontsize=9)
        ax.set_yticks(range(2 * n_top))
        ax.set_yticklabels(all_gene_names, fontsize=9)
        ax.set_xlabel('Associated Weight', fontsize=11)
        ax.set_title(f"{name} "f'Archetype_{i+1} - Top {n_top} Positive & Negative Genes', fontsize=13, fontweight='bold')
        ax.axvline(x=0, color='black', linestyle='-', alpha=0.5, linewidth=0.7)
        ax.grid(axis='x', alpha=0.2)
        ax.set_xlim(x_lim)

    # Hide any unused subplots
    for j in range(n_archetypes, n_rows * n_cols):
        fig.delaxes(axes[j])

    plt.tight_layout()
    return fig, axes

In [ ]:
plot_neg_pos_genes_per_archetype(archetype_refseqID, var_names, n_top=5)

### Mapping gene functinos
Using BacDrop provided maps of refseqidf to gene to function

In [ ]:
import pandas as pd

In [ ]:
def load_gene_mapping_simple(mapping_file_path):
    """
    Load the gene mapping from BacDrop_gene_mapping.csv, converting all 'cds-WP_' to 'cds-WP-'
    """
    mapping_df = pd.read_csv(mapping_file_path)
    
    # Create a dictionary for easy lookup
    gene_mapping = {}
    
    for _, row in mapping_df.iterrows():
        refseq_id = row['RefSeq_ID']
        gene_name = row['Gene']
        product = row['Product']
        
        # Convert all 'cds-WP_' to 'cds-WP-'
        if isinstance(refseq_id, str) and refseq_id.startswith('cds-WP_'):
            refseq_id = refseq_id.replace('cds-WP_', 'cds-WP-')
        
        gene_mapping[refseq_id] = {
            'gene_name': gene_name,
            'product': product
        }
    
    return gene_mapping


In [ ]:
gene_mapping = load_gene_mapping_simple(r"C:\Users\DG1\Desktop\DALLAB\Experimenting\Analysis\BacDrop_gene_mapping.csv")

In [ ]:
print(gene_mapping)


In [ ]:
# Create a DataFrame from archetype_refseqID so that rows and columns are labeled
# archetype_refseqID is now shape (n_archetypes, n_genes)
# We want the DataFrame to be archetype x gene (rows: archetypes, columns: genes)
df_archetypes = pd.DataFrame(
    archetype_refseqID,
    index=[f"Archetype {i+1}" for i in range(archetype_refseqID.shape[0])],
    columns=var_names
)

# Show the DataFrame
print(df_archetypes)


In [ ]:
# For each RefSeq_ID in the columns of df_archetypes, check if it exists in gene_mapping
in_gene_mapping = [refseq in gene_mapping for refseq in df_archetypes.columns]
in_gene_mapping_series = pd.Series(in_gene_mapping, index=df_archetypes.columns, name='In_Gene_Mapping')

# Optionally, print summary of matches
num_matched = sum(in_gene_mapping)
num_total = len(df_archetypes.columns)
print(f"{num_matched} out of {num_total} RefSeq_IDs in df_archetypes columns are found in gene_mapping.")

# Show the first few columns to verify
print(in_gene_mapping_series.head())



In [ ]:
# Transpose df_archetypes so that rows = genes (RefSeq_IDs), columns = archetypes
df_genes = df_archetypes.T

# Add gene_name and product columns
def get_gene_name(refseq):
    if refseq in gene_mapping:
        return gene_mapping[refseq].get('gene_name', None)
    return None

def get_product(refseq):
    if refseq in gene_mapping:
        return gene_mapping[refseq].get('product', None)
    return None

df_genes['gene_name'] = [get_gene_name(refseq) for refseq in df_genes.index]
df_genes['product'] = [get_product(refseq) for refseq in df_genes.index]

# Optionally, show a preview of the annotated gene x archetype matrix
print("Gene x Archetype matrix with annotation (first 5 rows):")
print(df_genes.head())


In [ ]:
# print  gene tables for each archetype using the orientation of df_genes
print("\n" + "="*80)
print("ANNOTATED GENE TABLES FOR EACH ARCHETYPE (using df_genes orientation)")
print("="*80)

# df_genes: rows = RefSeq_IDs, columns = archetypes + gene_name + product
# We'll iterate over archetype columns (excluding 'gene_name' and 'product')

archetype_cols = [col for col in df_genes.columns if col not in ['gene_name', 'product']]

for archetype in archetype_cols:
    print(f"\n{archetype.upper()} - ANNOTATED GENE TABLE")
    print("="*60)
    
    # Get the weights for this archetype (column: archetype, index: RefSeq_IDs)
    weights = df_genes[archetype]
    # Sort RefSeq_IDs by weight
    sorted_refseqs = weights.sort_values(ascending=False)
    
    # Top 5 positive genes
    print("\nTOP 5 POSITIVE GENES:")
    print("-" * 60)
    print(f"{'RefSeq_ID':<25} | {'Gene Name':<20} | {'Product':<40}")
    print("-" * 60)
    for refseq in sorted_refseqs.head(5).index:
        gene_name = df_genes.loc[refseq, 'gene_name'] if pd.notnull(df_genes.loc[refseq, 'gene_name']) else 'Unknown'
        product = df_genes.loc[refseq, 'product'] if pd.notnull(df_genes.loc[refseq, 'product']) else 'Unknown'
        print(f"{refseq:<25} | {gene_name:<20} | {str(product)[:40]:<40}")
    
    # Top 5 negative genes
    print("\nTOP 5 NEGATIVE GENES:")
    print("-" * 60)
    print(f"{'RefSeq_ID':<25} | {'Gene Name':<20} | {'Product':<40}")
    print("-" * 60)
    for refseq in sorted_refseqs.tail(5).index:
        gene_name = df_genes.loc[refseq, 'gene_name'] if pd.notnull(df_genes.loc[refseq, 'gene_name']) else 'Unknown'
        product = df_genes.loc[refseq, 'product'] if pd.notnull(df_genes.loc[refseq, 'product']) else 'Unknown'
        print(f"{refseq:<25} | {gene_name:<20} | {str(product)[:40]:<40}")
    
    print()

### Plotting expression of marker genes 
BacDrop identifed 3 key genes of related functino in the paper: recA, IS5-like and ibpB 

These viusalisations enable analysis of the extent to which the arhcetypes and latent space capture relationshisp across these marker genes

In [ ]:
#accessing the expressino fo the marker genes in the input data 

# map gene names to RefSeq IDs using df_archetypes
genes_of_interest = ['recA', 'IS5-like', 'ibpB']
gene_id_map = {}
for gene in genes_of_interest:
    matches = df_archetypes[df_archetypes['gene_name'].str.lower() == gene.lower()]
    if not matches.empty:
        gene_id_map[gene] = matches.index[0]
    else:
        print(f"Warning: Gene {gene} not found in df_archetypes.")

print("Gene to RefSeq mapping used:", gene_id_map)

# get indices of these genes in adata.var_names
gene_indices = []
for gene in genes_of_interest:
    refseq = gene_id_map.get(gene)
    if refseq is not None and refseq in adata.var_names:
        gene_indices.append(list(adata.var_names).index(refseq))
    else:
        print(f"Warning: RefSeq {refseq} for gene {gene} not found in adata.var_names.")

# extract expression for these genes from adata.X 

expr_matrix = np.array(adata.X[:, gene_indices])

Plotting the expression on 3D plots 

In [ ]:


# prepare 3D coordinates for each cell in latent space
cell_coords_3d = data_latent_3d  # shape: (n_cells, 3)

# plot: 3D scatter, colored by each gene's expression, and overlay simplex using simplex_plot
fig = plt.figure(figsize=(18, 5))
for i, gene in enumerate(genes_of_interest):
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    # Plot the cells, colored by gene expression
    from matplotlib.colors import LinearSegmentedColormap

    # Create a custom colormap: light purple -> yellow-green -> vibrant green
    custom_cmap = LinearSegmentedColormap.from_list(
        "custom_viridis_gradient",
        [
            (0.0, "#e0bbff"),   # very light purple
            (0.4, "#baffc9"),  # light yellow-green
            (0.7, "#39ff14"),  # neon green (very vibrant)
            (1.0, "#008f11"),  # deep vibrant green
        ]
    )

    sc = ax.scatter(
        cell_coords_3d[:, 0], cell_coords_3d[:, 1], cell_coords_3d[:, 2],
        c=expr_matrix[:, i], cmap=custom_cmap, s=5, alpha=0.2
    )
    
    # Overlay the simplex using the helper function
    simplex_plot(ax, archetypes_latent_3d, edgecolor='blue', label='Simplex', linewidth=2)
    # ax.scatter(archetypes_latent_3d[:, 0], archetypes_latent_3d[:, 1], archetypes_latent_3d[:, 2], c='red', s=60, marker='*', label='Archetypes')

    # Add numbers to the archetype vertices (and ensure they're added to the simplex vertices)
    for j, (x, y, z) in enumerate(archetypes_latent_3d):
        ax.scatter(x, y, z, c='red', s=80, marker='*', edgecolor='k', zorder=10)  # emphasize vertex
        # Draw a white sphere for the label background (simulate with a larger white marker)
        ax.scatter(x, y, z, c='white', s=200, marker='o', edgecolor='none', zorder=11, alpha=1)
        ax.text(x, y, z, f"{j+1}", color='black', fontsize=14, fontweight='bold', ha='center', va='center', zorder=12)

    ax.set_title(f"Cells colored by {gene} expression")
    ax.set_xlabel('Latent 1')
    ax.set_ylabel('Latent 2')
    ax.set_zlabel('Latent 3')
    plt.colorbar(sc, ax=ax, shrink=0.6, label=f"{gene} expression")
fig.suptitle(f"{name}: 3D Simplex Plot: Cells colored by gene expression (recA, IS5-like, ibpB)", fontsize=16)
plt.tight_layout()
plt.show()

Plotting the expression on 2D plots 

In [ ]:
# orepare 2D coordinates for each cell in latent space
cell_coords_2d = data_latent_2d  # shape: (n_cells, 2)
archetypes_latent_2d = archetypes_latent_2d  # shape: (n_archetypes, 2)

# plot: 2D scatter, colored by each gene's expression, and overlay simplex using simplex_plot
fig = plt.figure(figsize=(18, 5))
for i, gene in enumerate(genes_of_interest):
    ax = fig.add_subplot(1, 3, i+1)
    # Plot the cells, colored by gene expression
    from matplotlib.colors import LinearSegmentedColormap

    # Create a custom colormap: light purple -> yellow-green -> vibrant green
    custom_cmap = LinearSegmentedColormap.from_list(
        "custom_viridis_gradient",
        [
            (0.0, "#e0bbff"),   # very light purple
            (0.4, "#baffc9"),  # light yellow-green
            (0.7, "#39ff14"),  # neon green (very vibrant)
            (1.0, "#008f11"),  # deep vibrant green
        ]
    )

    sc = ax.scatter(
        cell_coords_2d[:, 0], cell_coords_2d[:, 1],
        c=expr_matrix[:, i], cmap=custom_cmap, s=5, alpha=0.2
    )
    
    # Overlay the simplex using the helper function
    simplex_plot(ax, archetypes_latent_2d, edgecolor='blue', label='Simplex', linewidth=2)
    for j, (x, y) in enumerate(archetypes_latent_2d):
        ax.scatter(x, y, c='red', s=80, marker='*', edgecolor='k', zorder=10)  # emphasize vertex
        # Draw a white circle for the label background
        from matplotlib.patches import Circle
        circ = Circle((x, y), 0.05, color='white', zorder=11, linewidth=0)
        ax.add_patch(circ)
        ax.text(x, y, f"{j+1}", color='black', fontsize=14, fontweight='bold', ha='center', va='center', zorder=12)

    ax.set_title(f"Cells colored by {gene} expression")
    ax.set_xlabel('Latent 1')
    ax.set_ylabel('Latent 2')
    plt.colorbar(sc, ax=ax, shrink=0.6, label=f"{gene} expression")
fig.suptitle(f"{name}: 2D Simplex Plot: Cells colored by gene expression (recA, IS5-like, ibpB)", fontsize=16)
plt.tight_layout()
plt.show()